In [ ]:
# WHY BAYESIAN GP (PyMC) > CLASSICAL GP (scikit-learn) > KRIGING
# differ on how they handle hyperparameters

# bayes integrate over posterior (hyperparam uncertianity)

# both ignore hyperparam uncertainity
#opt marginal likelihood
# point estimate variogram

import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

x_obs = np.array([0.0, 1.0, 2.0, 4.0, 5.0])
# sim input data

y_obs = np.array([35.0, 28.0, 42.0, 31.0, 38.0])  # RMR values

# Sovereign Standard Scaling: Prevents 'Funnel' geometry and unit mismatch
X_mean, X_std = x_obs.mean(), x_obs.std()
y_mean, y_std = y_obs.mean(), y_obs.std()

X = ((x_obs - X_mean) / X_std)[:, None] # Shape: (5, 1) # pymc gp expect 2d input shape (n, input dim)
y = (y_obs - y_mean) / y_std            # Shape: (5,)


# define gp model
with pm.Model() as gp_model:
    # mu: Expected mean. Added to handle non-zero RMR baseline.
    mu = pm.Normal("mu", mu=0, sigma=0.5)

    ell = pm.Gamma("ell", alpha=2, beta=1)
    # mean is beta/ (alpha-1)
    # we want ell to be from o to 5 but pushed a bit further from 0 or else overfit

    # signal amplitiude ~ sill
    # to model sigma of 8,  η ∈ [2, 20] is reasonable
    # HalfCauchy replaced with HalfNormal for stability on thin data
    eta = pm.HalfNormal("eta", sigma=1.0)
    #? beta

    # noise, measruement error in rmr (epistemic)
    sigma = pm.HalfNormal("sigma", sigma=1.0)
    # HalfNormal with σ=5 puts mass on [0, 10]

    # input_dim=1: 1D input (chainage along tunnel)
    # ls=ell: length-scale is our random variable, not a fixed value
    mean_func = pm.gp.mean.Constant(c=mu)
    cov_func = eta**2 * pm.gp.cov.Matern32(input_dim=1, ls=ell)

    gp = pm.gp.Marginal(mean_func=mean_func, cov_func=cov_func)
    y_target = gp.marginal_likelihood("y_obs", X=X, y=y, sigma=sigma) # sim y obs and y, confuses

    # this line creating cov matrix, log probab compute and explore via differentiation

#Free parameters: mu (mean), ℓ (length-scale), η (amplitude), σ (noise); priors

#Kernel:           η² × Matérn-3/2(ℓ)")
#Likelihood:       y ~ N(mu, K + σ²I)

# run prior pred, diagnostics here but this is more about gp

In [ ]:
# 3. HIGH-SIGNAL SAMPLING
with gp_model:
    # Use standard NUTS (removing numpyro) for better reliability on small models
    trace = pm.sample(draws=2000, tune=2000, chains=4, target_accept=0.99, random_seed=42)

# Diagnostic Summary
var_names = ["mu", "ell", "eta", "sigma"]
print(az.summary(trace, var_names=var_names))

az.plot_trace(trace, var_names=var_names)
plt.suptitle("GP Hyperparameter Traces (High-Reliability Logic)", fontsize=14)
plt.tight_layout()
plt.show()

# Audit
rhat = az.rhat(trace, var_names=var_names)
print(f"\nR-hat values:")
for var in var_names:
    rh = float(rhat[var].values)
    status = "✓" if rh < 1.05 else "✗ CRITICAL" # Allow 1.05 for very thin data
    print(f"  {var}: {rh:.4f} {status}")

# Scale Recovery
post_mu = (trace.posterior["mu"] * y_std + y_mean).mean().values
print(f"\nFinal Forensic Mean RMR: {post_mu:.2f}")

In [ ]:
# 5. POSTERIOR PREDICTION: Visualizing Uncertainty
# Forecasting RMR across the tunnel chainage

x_new = np.linspace(-1, 6, 100) # Extended range
X_new = ((x_new - X_mean) / X_std)[:, None]

with gp_model:
    # Predictive distribution for the 'latent' f (smooth) and 'observed' y (noisy)
    f_pred = gp.conditional("f_pred", X_new)
    # Sampling from conditional
    post_pred = pm.sample_posterior_predictive(trace, var_names=["f_pred"])

# Recover scale
f_samples = post_pred.posterior_predictive["f_pred"].values.reshape(-1, 100)
f_samples_orig = (f_samples * y_std) + y_mean

# Visualization
plt.figure(figsize=(10, 5))
mu_f = f_samples_orig.mean(axis=0)
std_f = f_samples_orig.std(axis=0)

plt.plot(x_new, mu_f, label="Predictive Mean", color='C0')
plt.fill_between(x_new, mu_f - 2*std_f, mu_f + 2*std_f, alpha=0.3, label="95% Credible Interval")
plt.scatter(x_obs, y_obs, color='red', zorder=5, label="Observations (Drill Logs)")

plt.title("Himalayan GP: RMR Spatial Uncertainty", fontsize=14)
plt.xlabel("Chainage (m)")
plt.ylabel("RMR Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Prediction complete. Mean RMR at 3.0m: {mu_f[np.abs(x_new-3).argmin()]:.2f}")

In [ ]:
# out of sample prediction
# .conditional() computes the posterior predictive distribution

X_new = np.linspace(-0.5, 5.5, 200)[:, None]  # Shape: (200, 1)

with gp_model:
    # Add the conditional distribution to the model
    f_pred = gp.conditional("f_pred", Xnew=X_new)

    # Sample from the posterior predictive
    ppc = pm.sample_posterior_predictive(
        trace,
        var_names=["f_pred"],
        random_seed=42,
    )

# Extract predictions
f_samples = ppc.posterior_predictive["f_pred"].values  # Shape: (chains, draws, 200)
f_samples = f_samples.reshape(-1, 200)  # Flatten chains: (4000, 200)

# summary statistics
f_mean = f_samples.mean(axis=0)
f_std = f_samples.std(axis=0)
f_q025 = np.percentile(f_samples, 2.5, axis=0)
f_q975 = np.percentile(f_samples, 97.5, axis=0)

print(f"Posterior predictive shape: {f_samples.shape}")
#Each of 4000 samples is a 200-point function realization
#The mean and percentiles summarize 4000 plausible RMR profiles

In [ ]:
# Full Bayesian GP Posterior

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Posterior predictive with credible band
ax = axes[0]
# Plot 50 random posterior samples (thin gray lines)
for i in range(50):
    idx = np.random.randint(0, f_samples.shape[0])
    ax.plot(X_new.flatten(), f_samples[idx], color='steelblue',
            alpha=0.05, linewidth=0.8)

# Credible band and mean
ax.fill_between(X_new.flatten(), f_q025, f_q975,
                alpha=0.3, color='steelblue', label='95% credible band')
ax.plot(X_new.flatten(), f_mean, 'b-', linewidth=2, label='Posterior mean')
ax.scatter(x_obs, y_obs, c='red', s=100, zorder=5, edgecolors='black',
           label='Boreholes', linewidth=1.5)
ax.set_xlabel('Chainage (km)', fontsize=12)
ax.set_ylabel('RMR Value', fontsize=12)
ax.set_title('Bayesian GP: Full Posterior (PyMC)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: Posterior uncertainty (std dev)
ax = axes[1]
ax.fill_between(X_new.flatten(), 0, f_std, alpha=0.4, color='orange')
ax.plot(X_new.flatten(), f_std, 'darkorange', linewidth=2)
for xb in x_obs:
    ax.axvline(x=xb, color='red', linestyle='--', alpha=0.3)
ax.set_xlabel('Chainage (km)', fontsize=12)
ax.set_ylabel('Posterior Std Dev (RMR units)', fontsize=12)
ax.set_title('Uncertainty Map: WHERE Do We Not Know?', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


print("\n=== POINT PREDICTIONS ===")
for chainage in [0.5, 1.5, 3.0, 4.5]:
    idx = np.argmin(np.abs(X_new.flatten() - chainage))
    print(f"  Chainage {chainage:.1f} km: RMR = {f_mean[idx]:.1f} ± {f_std[idx]:.1f} "
          f"(95% CI: [{f_q025[idx]:.1f}, {f_q975[idx]:.1f}])")

# uncertainty is LOWEST near boreholes, HIGHEST between them.
#t chainage 3.0 km (no nearby borehole), the band is WIDEST.
#  This is the GP telling you: 'I don't have data here. Drill or accept the risk.